In [23]:
import json_tricks
answer = {}


inputs = json_tricks.load(open('inputs/inputs.json', 'r'))

# Backpropagation Practice

You are given a graph of operations:

<img src="imgs/task_2_4_graph_02.png" width="400" height="300" />

# Task 1

Write a function that calculates the value of this graph for any given input vector `x` and set of coefficients $b_1, b_2, c_1, c_2$ packed as vector of weights `w`.
In this function also return all the $z$ values.

In [24]:
import numpy as np

## YOU CAN DEFINE ANY FUNCTIONS YOU NEED ##
## YOUR CODE HERE ##

def graph_value(x, w):

    def sigmoid(x):
        return 1/(1+np.exp(-x))

    b1, b2, c1, c2 = w

    z1 = b1 + x
    z2 = b2 + x
    z3 = sigmoid(z1)
    z4 = sigmoid(z2)
    z5 = np.tanh(z2)
    z7 = z1 * z4
    z6 = c2 * z5
    z8 = c1*z7
    z9 = z3*z6
    y = z9+z8

    z = {
        'z1': z1,
        'z2': z2,
        'z3': z3,
        'z4': z4,
        'z5': z5,
        'z6': z6,
        'z7': z7,
        'z8': z8,
        'z9': z9
    }
    ## YOUR CODE HERE ##
    return y, z

In [25]:
answer['graph_value'] = [graph_value(**input) for input in inputs]

# Task 2

In this graph, find all direct derivatives of all the values over $z$ or $x$ or $w$.

For example, if $y = z_8 + z_9$, you need to find only $\partial_{z_9} y$ and $\partial_{z_8} y$ for that operation.

You have to do that for all the intermediate signals.

Write the result in the form of a dictionary, for example, $\partial_{z_9} y$:

```
['dy_dz9'] = 1
```

In [26]:
def graph_derivatives(x, w):

    _, z = graph_value(x, w)
    b1, b2, c1, c2 = w
    z1 = z['z1']
    z2 = z['z2']
    z3 = z['z3']
    z4 = z['z4']
    z5 = z['z5']
    z6 = z['z6']
    z7 = z['z7']
    z8 = z['z8']
    z9 = z['z9']

    res = {}
    res['dz1_db1'] = 1.0
    res['dz1_dx'] = 1.0

    res['dz2_db2'] = 1.0
    res['dz2_dx'] = 1.0

    res['dz3_dz1'] = z3 * (1 - z3)
    res['dz4_dz2'] = z4 * (1 - z4)
    res['dz5_dz2'] = 1 - z5**2

    res['dz7_dz1'] = z4
    res['dz7_dz4'] = z1

    res['dz6_dc2'] = z5
    res['dz6_dz5'] = c2

    res['dz8_dc1'] = z7
    res['dz8_dz7'] = c1

    res['dz9_dz3'] = z6
    res['dz9_dz6'] = z3

    res['dy_dz9'] = 1.0
    res['dy_dz8'] = 1.0
    return res

In [27]:
answer['graph_derivatives'] = [graph_derivatives(**input) for input in inputs]

# Task 3

Using the values of the derivatives, calculated above:
- extract a dictionary of values that are needed to calculate $\partial_{c_1} y$
- calculate that derivative

In [28]:
def dy_dc1(x, w):

    derivs = graph_derivatives(x, w)
    
    selected_derivs = {
        'dy_dz8': derivs['dy_dz8'], 
        'dz8_dc1': derivs['dz8_dc1']
    }
    
    dy_dc1 = selected_derivs['dy_dz8'] * selected_derivs['dz8_dc1']
    
    return selected_derivs, dy_dc1

In [29]:
answer['dy_dc1'] = [dy_dc1(**input) for input in inputs]

# Task 4
Using the values of the derivatives, calculated above:
- extract a dictionary of values that are needed to calculate $\partial_{c_2} y$
- calculate that derivative

In [30]:
def dy_dc2(x, w):
    derivs = graph_derivatives(x, w)
    
    selected_derivs = {
        'dy_dz9': derivs['dy_dz9'], 
        'dz9_dz6': derivs['dz9_dz6'],
        'dz6_dc2': derivs['dz6_dc2']
    }
    
    dy_dc2 = selected_derivs['dy_dz9'] * selected_derivs['dz9_dz6'] * selected_derivs['dz6_dc2']
    
    return selected_derivs, dy_dc2

In [31]:
answer['dy_dc2'] = [dy_dc2(**input) for input in inputs]

# Task 5
Using the values of the derivatives, calculated above:
- extract a dictionary of values that are needed to calculate $\partial_{b_1} y$
- calculate that derivative

In [32]:
def dy_db1(x, w):
    derivs = graph_derivatives(x, w)
    
    selected_derivs = {
        'dy_dz9': derivs['dy_dz9'],      
        'dz9_dz3': derivs['dz9_dz3'],      
        'dz3_dz1': derivs['dz3_dz1'], 
        'dz1_db1': derivs['dz1_db1'],      
        'dy_dz8': derivs['dy_dz8'], 
        'dz8_dz7': derivs['dz8_dz7'], 
        'dz7_dz1': derivs['dz7_dz1']  
    }
    
    path1 = (selected_derivs['dy_dz9'] * 
             selected_derivs['dz9_dz3'] * 
             selected_derivs['dz3_dz1'] * 
             selected_derivs['dz1_db1'])
    
    path2 = (selected_derivs['dy_dz8'] * 
             selected_derivs['dz8_dz7'] * 
             selected_derivs['dz7_dz1'] * 
             selected_derivs['dz1_db1'])
    
    dy_db1 = path1 + path2
    
    return selected_derivs, dy_db1

In [33]:
answer['dy_db1'] = [dy_db1(**input) for input in inputs]

# Task 6
Using the values of the derivatives, calculated above:
- extract a dictionary of values that are needed to calculate $\partial_{b_2} y$
- calculate that derivative

In [34]:
def dy_db2(x, w):
    derivs = graph_derivatives(x, w)
    
    selected_derivs = {
        'dy_dz8': derivs['dy_dz8'],        
        'dz8_dz7': derivs['dz8_dz7'], 
        'dz7_dz4': derivs['dz7_dz4'],
        'dz4_dz2': derivs['dz4_dz2'],
        'dz2_db2': derivs['dz2_db2'],
        'dy_dz9': derivs['dy_dz9'],
        'dz9_dz6': derivs['dz9_dz6'], 
        'dz6_dz5': derivs['dz6_dz5'],
        'dz5_dz2': derivs['dz5_dz2'] 
    }
    
    path1 = (selected_derivs['dy_dz8'] * 
             selected_derivs['dz8_dz7'] * 
             selected_derivs['dz7_dz4'] * 
             selected_derivs['dz4_dz2'] * 
             selected_derivs['dz2_db2'])
    
    path2 = (selected_derivs['dy_dz9'] * 
             selected_derivs['dz9_dz6'] * 
             selected_derivs['dz6_dz5'] * 
             selected_derivs['dz5_dz2'] * 
             selected_derivs['dz2_db2'])
    
    dy_db2 = path1 + path2
    
    return selected_derivs, dy_db2

In [35]:
answer['dy_db2'] = [dy_db2(**input) for input in inputs]

In [36]:
json_tricks.dump(answer, '.answer.json')

'{"graph_value": [[{"__ndarray__": [-0.4080193630022569, -0.2243997267055186], "dtype": "float64", "shape": [2]}, {"z1": {"__ndarray__": [1.1444026911119252, 0.5094242369295079], "dtype": "float64", "shape": [2]}, "z2": {"__ndarray__": [2.019744009419258, 1.3847655552368408], "dtype": "float64", "shape": [2]}, "z3": {"__ndarray__": [0.7584870610795774, 0.6246714924681956], "dtype": "float64", "shape": [2]}, "z4": {"__ndarray__": [0.8828545365576449, 0.7997552788670441], "dtype": "float64", "shape": [2]}, "z5": {"__ndarray__": [0.9653962812375015, 0.8820139250377154], "dtype": "float64", "shape": [2]}, "z6": {"__ndarray__": [-0.22603494753900388, -0.20651205639513331], "dtype": "float64", "shape": [2]}, "z7": {"__ndarray__": [1.0103411074969404, 0.40741472266718975], "dtype": "float64", "shape": [2]}, "z8": {"__ndarray__": [-0.23657477994212137, -0.09539753222449449], "dtype": "float64", "shape": [2]}, "z9": {"__ndarray__": [-0.17144458306013552, -0.12900219448102412], "dtype": "float64